# Single Agent Systems & Agent Pipelines - Quiz

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response


## Step 1 – Install Required Libraries

In [1]:
!pip -q install transformers torch

## Step 2 – Import Libraries

In [2]:
import re
import json
import logging
from datetime import datetime
from transformers import pipeline

## Step 3 – Configure Logging

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("SmartAgent")

print("Logging configured successfully.")

Logging configured successfully.


## Step 4 – Load FLAN-T5 Model

In [4]:
def general_response_tool(query):

    responses = {
        "what is ai": "Artificial Intelligence is the simulation of human intelligence by machines.",
        "what is machine learning": "Machine Learning is a subset of AI that learns from data.",
        "what is python": "Python is a popular programming language used for AI and Data Science.",
        "what is cloud computing": "Cloud Computing provides computing services over the Internet."
    }

    query = query.lower()

    for key in responses:
        if key in query:
            return {"response": responses[key]}

    return {
        "response": "Sorry, I don't have an answer for that question."
    }

## Step 5 – Calculator Tool

In [5]:
def calculator_tool(expression):
    """
    Safely evaluates basic mathematical expressions.
    """
    try:
        # Allow only numbers, operators, parentheses and spaces
        if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
            return {"error": "Invalid mathematical expression"}

        result = eval(expression, {"__builtins__": {}}, {})
        return {"result": result}

    except Exception as e:
        return {"error": str(e)}

## Step 6 – Keyword Extraction Tool

In [6]:
def keyword_tool(text):
    """
    Extracts important keywords from a sentence.
    """
    stopwords = {
        "the", "is", "a", "an", "and", "or", "of", "to", "in",
        "for", "on", "with", "this", "that", "it", "are"
    }

    words = re.findall(r"[A-Za-z]+", text.lower())

    keywords = []
    for word in words:
        if word not in stopwords and len(word) > 3:
            keywords.append(word)

    # Remove duplicates while preserving order
    unique_keywords = list(dict.fromkeys(keywords))

    return {"keywords": unique_keywords[:10]}

## Step 7 – Date & Time Tool (Bonus)

In [7]:
def datetime_tool():
    now = datetime.now()

    return {
        "date": now.strftime("%Y-%m-%d"),
        "time": now.strftime("%H:%M:%S")
    }

## Step 8 – General Response Tool

In [8]:
def general_response_tool(query):
    """
    Uses FLAN-T5 to answer general questions.
    """
    try:
        response = generator(query, max_length=80)[0]["generated_text"]
        return {"response": response}

    except Exception as e:
        return {"error": str(e)}

## Step 9 – Intent Detection

In [9]:
def detect_intent(query):
    query_lower = query.lower()

    # Date & Time intent
    datetime_keywords = [
        "date", "time", "today", "current time", "what time"
    ]

    if any(k in query_lower for k in datetime_keywords):
        return "datetime"

    # Keyword extraction intent
    keyword_keywords = [
        "extract keywords",
        "keywords from",
        "important words"
    ]

    if any(k in query_lower for k in keyword_keywords):
        return "keyword_extraction"

    # Math intent
    math_pattern = r"[0-9]+[\s]*[+\-*/][\s]*[0-9]+"

    if re.search(math_pattern, query):
        return "math"

    # Default intent
    return "general"

## Step 10 – Single-Agent Smart Assistant

In [10]:
def smart_agent(query):
    logger.info(f"Received query: {query}")

    try:
        intent = detect_intent(query)

        # ---------------- MATH ----------------
        if intent == "math":

            expression_match = re.search(
                r"([0-9+\-*/(). ]+)",
                query
            )

            expression = expression_match.group(1).strip()

            tool_output = calculator_tool(expression)

            if "error" in tool_output:
                return {
                    "query": query,
                    "intent": intent,
                    "tool_used": "Calculator Tool",
                    "status": "error",
                    "message": tool_output["error"]
                }

            return {
                "query": query,
                "intent": intent,
                "tool_used": "Calculator Tool",
                "result": tool_output["result"],
                "status": "success"
            }

        # ----------- KEYWORD EXTRACTION -----------
        elif intent == "keyword_extraction":

            clean_text = query
            clean_text = clean_text.replace("extract keywords from", "")
            clean_text = clean_text.replace("keywords from", "")
            clean_text = clean_text.strip()

            tool_output = keyword_tool(clean_text)

            return {
                "query": query,
                "intent": intent,
                "tool_used": "Keyword Tool",
                "keywords": tool_output["keywords"],
                "status": "success"
            }

        # ---------------- DATE & TIME ----------------
        elif intent == "datetime":

            tool_output = datetime_tool()

            return {
                "query": query,
                "intent": intent,
                "tool_used": "Date & Time Tool",
                "date": tool_output["date"],
                "time": tool_output["time"],
                "status": "success"
            }

        # ---------------- GENERAL ----------------
        else:

            tool_output = general_response_tool(query)

            if "error" in tool_output:
                return {
                    "query": query,
                    "intent": intent,
                    "tool_used": "General Response Tool",
                    "status": "error",
                    "message": tool_output["error"]
                }

            return {
                "query": query,
                "intent": intent,
                "tool_used": "General Response Tool",
                "response": tool_output["response"],
                "status": "success"
            }

    except Exception as e:

        logger.error(f"Agent error: {str(e)}")

        return {
            "query": query,
            "intent": "unknown",
            "tool_used": "None",
            "status": "error",
            "message": str(e)
        }

## Step 11 – Test the Agent

In [11]:
test_queries = [
    "25 * 12",
    "100 / 4 + 5",
    "Extract keywords from Artificial Intelligence is transforming healthcare and education.",
    "What is Machine Learning?",
    "Explain cloud computing in simple words.",
    "What is the current date and time?"
]

for q in test_queries:
    print("\n" + "="*80)
    result = smart_agent(q)
    print(json.dumps(result, indent=4))


{
    "query": "25 * 12",
    "intent": "math",
    "tool_used": "Calculator Tool",
    "result": 300,
    "status": "success"
}

{
    "query": "100 / 4 + 5",
    "intent": "math",
    "tool_used": "Calculator Tool",
    "result": 30.0,
    "status": "success"
}

{
    "query": "Extract keywords from Artificial Intelligence is transforming healthcare and education.",
    "intent": "keyword_extraction",
    "tool_used": "Keyword Tool",
    "keywords": [
        "extract",
        "artificial",
        "intelligence",
        "transforming",
        "healthcare",
        "education"
    ],
    "status": "success"
}

{
    "query": "What is Machine Learning?",
    "intent": "general",
    "tool_used": "General Response Tool",
    "status": "error",
    "message": "name 'generator' is not defined"
}

{
    "query": "Explain cloud computing in simple words.",
    "intent": "general",
    "tool_used": "General Response Tool",
    "status": "error",
    "message": "name 'generator' is not d

## Step 12 – Interactive Smart Assistant

In [12]:
print("Single-Agent Smart Assistant")
print("Type 'exit' to stop.\n")

while True:

    user_query = input("You: ")

    if user_query.lower() == "exit":
        print("Agent: Goodbye!")
        break

    result = smart_agent(user_query)

    print("\nAgent Output:")
    print(json.dumps(result, indent=4))
    print()

Single-Agent Smart Assistant
Type 'exit' to stop.

You: exit
Agent: Goodbye!


## Conclusion

This project successfully implements a **Single-Agent Agentic AI Pipeline**.

### Features Completed
- Intent detection
- Conditional routing
- Tool integration
- Structured JSON responses
- Error handling
- Logging
- Bonus Date & Time tool

### Tools Used
- Python
- Hugging Face Transformers
- FLAN-T5
- Regular Expressions
- JSON
- Logging Module